# OOD Detekció - Végső Eredmények és Metrikák Elemzése

Ebben a notebookban a betanított modelljeink (MLP, CNN, ResNet) teljesítményét értékeljük ki. A cél annak vizsgálata, hogy a modellek mennyire magabiztosak a saját eloszlásukból származó (In-Distribution - ID) adatokon, és hogyan viselkednek az ismeretlen (Out-of-Distribution - OOD) képeken.

A kiértékeléshez a következő metrikákat használjuk:
* **AUROC** (Area Under the Receiver Operating Characteristic curve)
* **AUPR** (Area Under the Precision-Recall curve - In és Out)
* **FPR@95TPR** (False Positive Rate at 95% True Positive Rate)

**Kutatási kérdések:**
1. Melyik a legoptimálisabb `Temperature Scaling (T)` paraméter a modellekhez?
2. Milyen zajszintnél (1-10) omlik össze a modellek detekciós képessége (AUROC < 60%)?

In [ ]:
import pandas as pd
import numpy as np
import os

# Ide kerülnek majd a valós eredmények a metrics.py kimenetéből. 
# Most teszteléshez mock adatokat használunk.
data = [
    {"Modell": "MLP", "Kísérlet": "Baseline", "Zaj_Szint": 0, "T": 1, "AUROC": 85.2, "AUPR-In": 86.1, "AUPR-Out": 84.5, "FPR95": 45.3},
    {"Modell": "MLP", "Kísérlet": "Baseline", "Zaj_Szint": 0, "T": 2, "AUROC": 88.1, "AUPR-In": 89.0, "AUPR-Out": 87.2, "FPR95": 38.1},
    
    {"Modell": "CNN", "Kísérlet": "Baseline", "Zaj_Szint": 0, "T": 1, "AUROC": 93.5, "AUPR-In": 94.2, "AUPR-Out": 92.8, "FPR95": 20.4},
    {"Modell": "CNN", "Kísérlet": "Baseline", "Zaj_Szint": 0, "T": 5, "AUROC": 95.8, "AUPR-In": 96.1, "AUPR-Out": 94.5, "FPR95": 15.2},
    {"Modell": "CNN", "Kísérlet": "Gauss-zaj", "Zaj_Szint": 4, "T": 1, "AUROC": 72.1, "AUPR-In": 75.0, "AUPR-Out": 70.1, "FPR95": 65.4},
    {"Modell": "CNN", "Kísérlet": "Gauss-zaj", "Zaj_Szint": 8, "T": 1, "AUROC": 52.3, "AUPR-In": 51.5, "AUPR-Out": 53.0, "FPR95": 92.1},
    
    {"Modell": "ResNet", "Kísérlet": "Baseline", "Zaj_Szint": 0, "T": 1, "AUROC": 96.2, "AUPR-In": 97.0, "AUPR-Out": 95.5, "FPR95": 12.5},
    {"Modell": "ResNet", "Kísérlet": "Gauss-zaj", "Zaj_Szint": 10, "T": 1, "AUROC": 65.4, "AUPR-In": 67.2, "AUPR-Out": 62.1, "FPR95": 45.3},
]

df = pd.DataFrame(data)
df  # Jupyterben ez a sor magától kirajzolja a szép táblázatot!

## 1. Az Optimális Temperature Scaling (T) Keresése

A [Hendrycks és Gimpel (2016)] cikk alapján a nyers softmax kimenetek eloszlásának "simítása" (Temperature Scaling) segíthet abban, hogy a modell ne legyen túlzottan magabiztos a rossz predikcióknál.
Ebben a lépésben megkeressük, hogy az In-Distribution adatokon (zajmentes Baseline) melyik `T` érték adta a legmagasabb AUROC pontszámot az egyes modelleknél.

In [ ]:
# Csak a Baseline (zajmentes) kísérleteket nézzük
baseline_df = df[df['Kísérlet'] == 'Baseline']

# Kikeressük azokat a sorokat, ahol a modellhez tartozó AUROC a maximális
idx = baseline_df.groupby(['Modell'])['AUROC'].idxmax()
optimal_t_df = baseline_df.loc[idx][['Modell', 'T', 'AUROC', 'FPR95']]

print("Optimális Temperature (T) értékek modellenként:")
optimal_t_df

## 2. Zaj-Robusztusság és Összeomlási Pontok

A modelleket különböző típusú és intenzitású (1-től 10-es skálán mozgó) zajoknak vetettük alá (Gauss, Egyenletes, Só-Bors). 
A véletlen tippelés (Random Guess) egy bináris klasszifikációs problémánál 50%-os AUROC-ot jelent. A következő blokk kiszámolja, hogy az egyes modellek melyik zajszintnél "adják fel", vagyis mikor esik a detekciós teljesítményük egy kritikus határ (60%) alá.

In [ ]:
collapse_threshold = 60.0
noise_df = df[df['Zaj_Szint'] > 0]

print(f"Modell összeomlási szintek (AUROC < {collapse_threshold}%):")
for name, group in noise_df.groupby(['Modell', 'Kísérlet']):
    modell, zaj_tipus = name
    collapsed = group[group['AUROC'] < collapse_threshold]
    
    if not collapsed.empty:
        first_collapse = collapsed.sort_values(by='Zaj_Szint').iloc[0]
        print(f"- {modell} | {zaj_tipus}: Szint {first_collapse['Zaj_Szint']} (AUROC: {first_collapse['AUROC']}%)")
    else:
        print(f"- {modell} | {zaj_tipus}: Kitartott a 10-es szintig!")

## 3. Eredmények Exportálása a README-hez

Hogy a projekt dokumentációja (README.md) is naprakész legyen, az összesített adatokat (Pandas DataFrame) a `tabulate` könyvtár segítségével formázott Markdown táblázattá alakítjuk, és kimentjük a `results/` mappába.

In [10]:
if not os.path.exists("results"):
    os.makedirs("results")
    
filepath = "../results/metrics/metrics_summary.md"
markdown_table = df.to_markdown(index=False)

with open(filepath, "w", encoding="utf-8") as f:
    f.write("# OOD Detekciós Eredmények (Baseline & Robusztusság)\n\n")
    f.write(markdown_table)
    
print(f"Markdown táblázat kimentve a README-hez: {filepath}")

Markdown táblázat kimentve a README-hez: ../results/metrics/metrics_summary.md
